# User-Configurable MCP Server Gateway

**Status:** Approved design; written specification awaiting review
**Design epic:** `bd-3ii8a`
**Source optimizations:** `sol_d55549d1521547bb` (injection scope, Z3 Optimize 4.16.0, complete), `sol_7dc84b7ffd7745bd` (transports, complete), `sol_c1ed58eb2162400e` (apply semantics, complete)
**Interim gate proofs:** `sol_145247a55fc74456`, `sol_f5597afb2f20434a`, `sol_3a062f0ef17d4361` (unsat), `sol_512c916a664a446b`, `sol_2d36692cdb9c43a1` (sat witnesses)
**Authoring fallback:** Notebook MCP was unavailable at authoring time; this notebook is hand-written and the two `ns_mermaid` cells below are authored-but-not-executed. The underlying obligations were verified through the solve MCP (evidence table in "Proof evidence"); the cells must be executed with the notebook cell runner to mint fresh proof hashes before implementation sign-off.


## Problem and grounded evidence

Users cannot add their own MCP servers today. The injected server sets are hardcoded:

- Brain sessions always receive `spur-mcp` (HTTP) plus the notebook stdio proxy (`brain_mcp_servers`, `crates/spur-core/src/notebook.rs:402`).
- Workers and direct exec receive only the curated `spur-worker-mcp` HTTP entry (`build_worker_mcp_servers_with`, `crates/spur-core/src/orchestrator/worker_mcp.rs:136`; dual token delivery in `assemble_worker_mcp_http_entry`).
- The ACP layer already models both transports: `McpServer::Http` / `McpServer::Stdio` (`spur-acp`).
- `/configure` persists one section at a time via the SAVE-APPLY `ConfigPatch` flow (`crates/spur-acp/src/config/mod.rs:631`; sections today: `agents`, `graph`, `tui`, `skills`), routed by `crates/spur-tui/src/commands/submit_router.rs:166` and rendered through `crates/spur-tui/src/configure_section.rs` / `settings_tui.rs`.

Goal: a user-configurable list of MCP servers, managed through a new `/configure mcp` section, injected into brain sessions.


## Architecture decision (Z3 Optimize, all termination: complete)

### 1. Injection scope — brain sessions only

Weights: curated_worker_boundary=5, minimal_v1_surface=3, worker_access_available=2, forward_compat_to_C=2 (`sol_d55549d1521547bb`).

| Candidate | boundary (5) | surface (3) | access (2) | fwd-compat (2) | Total |
|---|---:|---:|---:|---:|---:|
| **A brain only** | 1 | 1 | 1 | 1 | **12** |
| C opt-in workers | 1 | 0 | 1 | 1 | 9 |
| B brain+workers | 0 | 0 | 1 | 0 | 2 |

Workers keep the curated catalog; user servers reach interactive brain sessions only. C remains reachable later via an opt-in flag with no schema change.

### 2. Transports — both stdio and HTTP

Weights: stdio_coverage=4, http_coverage=3, minimal_ui_surface=1 (`sol_7dc84b7ffd7745bd`).

| Candidate | stdio (4) | http (3) | ui (1) | Total |
|---|---:|---:|---:|---:|
| **both** | 1 | 1 | 0 | **7** |
| stdio-only | 1 | 0 | 1 | 5 |
| http-only | 0 | 1 | 1 | 4 |

### 3. Apply semantics — next session

Weights: simple_impl_acp_protocol=4, no_restart_churn=3, immediate_effect=1 (`sol_c1ed58eb2162400e`). ACP delivers `mcp_servers` at `NewSessionRequest`; mid-session injection would force session restarts. SAVE-APPLY persists the config; the TUI must display an "applies to next session" notice.

Caveat recorded on the epic: these results are optimal under the stated preference weights, not universal proofs.


In [ ]:
flowchart TD
    SPEC["`@spec MCP-ENTRY-VALIDATION
@type Status = enum[rejected_reserved, rejected_duplicate, rejected_transport, rejected_payload, accepted]
@input name_is_reserved: Bool
@input name_is_unique: Bool
@input stdio_defined: Bool
@input http_defined: Bool
@input payload_nonempty: Bool
@output status: Status
@requires PRE: true`"]

    R_RESERVED["`@branch REJECT_RESERVED
@when name_is_reserved
@ensures RESERVED_STATUS: status = rejected_reserved`"]

    R_DUP["`@branch REJECT_DUPLICATE
@when not name_is_reserved and not name_is_unique
@ensures DUP_STATUS: status = rejected_duplicate`"]

    R_TRANSPORT["`@branch REJECT_TRANSPORT
@when not name_is_reserved and name_is_unique and (stdio_defined = http_defined)
@ensures TRANSPORT_STATUS: status = rejected_transport`"]

    R_PAYLOAD["`@branch REJECT_PAYLOAD
@when not name_is_reserved and name_is_unique and ((stdio_defined and (not http_defined)) or (http_defined and (not stdio_defined))) and not payload_nonempty
@ensures PAYLOAD_STATUS: status = rejected_payload`"]

    ACCEPT["`@branch ACCEPT
@when not name_is_reserved and name_is_unique and ((stdio_defined and (not http_defined)) or (http_defined and (not stdio_defined))) and payload_nonempty
@ensures ACCEPT_STATUS: status = accepted`"]

    CHECK["`@verify ENTRY_DETERMINISTIC: prove determinism
@verify ENTRY_COVERAGE: prove partition_coverage
@verify ENTRY_EXCLUSIVE: prove partition_exclusive
@verify STATUSES_REACHABLE: witness each status`"]

    SPEC --> R_RESERVED --> CHECK
    SPEC --> R_DUP --> CHECK
    SPEC --> R_TRANSPORT --> CHECK
    SPEC --> R_PAYLOAD --> CHECK
    SPEC --> ACCEPT --> CHECK


In [ ]:
flowchart TD
    SPEC["`@spec MCP-SESSION-INJECTION
@type Decision = enum[injected, skipped_worker, skipped_disabled, skipped_invalid]
@input session_is_brain: Bool
@input entry_enabled: Bool
@input entry_valid: Bool
@output status: Decision
@requires PRE: true`"]

    INJECT["`@branch INJECT
@when session_is_brain and entry_enabled and entry_valid
@ensures INJECT_STATUS: status = injected`"]

    WORKER["`@branch SKIP_WORKER
@when not session_is_brain
@ensures WORKER_STATUS: status = skipped_worker`"]

    DISABLED["`@branch SKIP_DISABLED
@when session_is_brain and not entry_enabled
@ensures DISABLED_STATUS: status = skipped_disabled`"]

    INVALID["`@branch SKIP_INVALID
@when session_is_brain and entry_enabled and not entry_valid
@ensures INVALID_STATUS: status = skipped_invalid`"]

    CHECK["`@verify INJECTION_DETERMINISTIC: prove determinism
@verify INJECTION_COVERAGE: prove partition_coverage
@verify INJECTION_EXCLUSIVE: prove partition_exclusive
@verify DECISIONS_REACHABLE: witness each status`"]

    SPEC --> INJECT --> CHECK
    SPEC --> WORKER --> CHECK
    SPEC --> DISABLED --> CHECK
    SPEC --> INVALID --> CHECK


## Proof evidence

Both gates reduce to: (a) the transport predicates `eq(stdio_defined, http_defined)` and `xor(stdio_defined, http_defined)` are mutually exclusive and exhaustive, (b) the remaining guards are chains over distinct propositional literals (`name_is_reserved`, `name_is_unique`, `payload_nonempty`; `session_is_brain`, `entry_enabled`, `entry_valid`), whose complementarity is definitional.

| Query | solve_id | Status | Meaning |
|---|---|---|---|
| eq(s,h) and xor(s,h) | `sol_145247a55fc74456` | unsat | transport predicates mutually exclusive (ENTRY_EXCLUSIVE crux) |
| not eq(s,h) and not xor(s,h) | `sol_f5597afb2f20434a` | unsat | transport predicates exhaustive (ENTRY_COVERAGE crux) |
| c1 and c4 (worker-skip and inject) | `sol_3a062f0ef17d4361` | unsat | INJECTION_EXCLUSIVE witness pair |
| accept guard | `sol_512c916a664a446b` | sat | model r=F,u=T,s=T,h=F,p=T (accepted reachable) |
| inject guard | `sol_2d36692cdb9c43a1` | sat | model brain=T (injected reachable) |

Retraction note: an initial single-tree encoding of ENTRY partition (`sol_18813881743a49c9`) returned sat; hand-evaluation of the returned model showed only one branch firing, i.e. the tree was mis-nested by the author. It was decomposed into the lemma queries above. Re-encoding lessons: decompose partitions into small auditable queries; the cell runner re-verifies the full obligations at execution time.

These are interim proofs over the abstracted Boolean facts. At implementation time, re-run against the real validation sites: uniqueness as `data_integrity.unique`, single-transport as `data_integrity.cardinality` or `configuration.selection_cardinality`, reserved-name exclusion as `configuration.excludes` over the concrete entry sets.


## Config schema and SAVE-APPLY flow

New section in `SpurConfig` (`spur-acp/src/config/mod.rs`), following the existing `agents` pattern:

```toml
[[mcp_servers.entries]]
name = "github"          # required, unique, not in RESERVED
enabled = true           # default true

[mcp_servers.entries.stdio]          # exactly one transport block
command = "npx"
args = ["-y", "@modelcontextprotocol/server-github"]
env = { GITHUB_TOKEN = "..." }

# alternative:
# [mcp_servers.entries.http]
# url = "https://mcp.example.com/sse"
# headers = { Authorization = "Bearer ..." }
```

- `McpServerEntry { name, enabled, transport: McpServerTransport }` where `McpServerTransport` is `Stdio { command, args, env }` xor `Http { url, headers }` (serde untagged with `#[serde(deny_unknown_fields)]`-style validation so both blocks is a load error).
- `ConfigPatch::McpServerUpsert { entry }` and `ConfigPatch::McpServerRemove { name }`, `section_id() = "mcp"`. `apply()` enforces the MCP-ENTRY-VALIDATION gate: unique names, reserved-name exclusion (`spur-mcp`, `notebook`, `spur-worker-mcp`), exactly one transport, non-empty command/url.
- Reserved names live next to `WORKER_MCP_SERVER_NAME` as a single const list.

## Injection

`brain_mcp_servers` (`crates/spur-core/src/notebook.rs:402`) gains the enabled+valid user entries appended **after** the fixed servers (`spur-mcp`, notebook proxy). The orchestrator passes the config at session creation. Worker paths are untouched (`build_worker_mcp_servers_with` keeps its curated single entry) — MCP-SESSION-INJECTION gate, `skipped_worker`.

## TUI

New `/configure mcp` section: server list (name, transport, enabled), add/edit/remove/toggle actions, stdio and HTTP forms. SAVE-APPLY persists through the existing one-section mutation flow (`crates/spur-tui/src/action.rs:136`, `app/mod.rs:157`); the section footer shows "applies to next session".

## Task decomposition (for the implementation plan DAG)

- **T1** `spur-acp`: config schema + `ConfigPatch` variants + validation + tests (round-trip serde, gate rejections).
- **T2** `spur-core`: `brain_mcp_servers` extension + orchestrator wiring + tests (depends on T1).
- **T3** `spur-tui`: `/configure mcp` section UI + router entry + tests (depends on T1; parallel with T2).

## Testing strategy

- RED-GREEN per repo convention: failing `test(...)` commit, then `fix(...)`.
- Unit: validation gate truth table (all five statuses), reserved list, serde round-trip including reject-on-both-transports.
- Integration: `brain_mcp_servers` returns fixed servers followed by enabled user entries; disabled/invalid entries excluded; worker dispatch unchanged.
- TUI: section render + SAVE-APPLY persist (follow existing settings_tui test harness).
- Solver: re-run catalog rules (`data_integrity.unique`, `configuration.selection_cardinality`, `configuration.excludes`) against concrete entry fixtures after implementation.

## Non-goals (v1)

- Worker/direct-exec injection of user servers (opt-in flag is a future extension; schema unchanged).
- Live mid-session re-injection or session restarts.
- Per-project server scope (single canonical `SpurConfig` location, same as `agents`).
- OAuth flow management for HTTP servers (headers carry tokens; users manage rotation).

## Risks

- Stdio servers hold secrets in `env`; same exposure class as the existing `agents` section — documented, not solved, in v1.
- Name collisions with future reserved names → validation rejects at persist time, fail-closed.
- A broken user server can slow session startup for clients that eagerly connect; mitigated by client-side laziness (out of scope) and the enabled toggle.
